# Introducción

En este cuaderno estaremos explorando el funcionamiento de los LLMs.

**¿Qué es un modelo de lenguaje?**

Un modelo de lenguaje (Language Model o LM) es una red neuronal entrenada para predecir cuál es el siguiente token más probable dado un contexto.

Por ejemplo, si el modelo recibe:

```text
The capital of France is
```

internamente intentará estimar las probabilidades de todos los tokens de su vocabulario.

| Token          | Probabilidad |
| -------------- | -----------: |
| `" Paris"`     |        92.3% |
| `" Lyon"`      |         2.1% |
| `" Marseille"` |         0.8% |
| ...            |          ... |

El modelo **no piensa en palabras completas**, sino en **tokens**.

---

**¿Qué es un token?**

Un token es la unidad mínima que procesa el modelo.

Dependiendo del tokenizer, un token puede ser:

* una palabra completa
* parte de una palabra
* un signo de puntuación
* un espacio
* incluso un carácter

Por ejemplo:

```text
The cat sat.
```

podría tokenizarse como:

```text
["The", " cat", " sat", "."]
```

Cada token tiene un identificador entero (ID).

Ejemplo:

```text
"The"      -> 345
" cat"     -> 812
" sat"     -> 1045
"."        -> 73
```

El modelo únicamente trabaja con estos IDs.

---

**¿Qué hace realmente un modelo?**

Cuando ejecutamos:

```python
outputs = model(**inputs)
```

el modelo devuelve una matriz llamada **logits**.

Su forma es:

```python
(batch_size, sequence_length, vocab_size)
```

Por ejemplo:

```python
torch.Size([1, 6, 151936])
```

Esto significa:

* 1 prompt
* 6 tokens de entrada
* un vocabulario de 151 936 tokens

---

**¿Qué son los logits?**

Los logits son puntuaciones (scores) sin normalizar para cada token del vocabulario.

No son probabilidades.

Por ejemplo:

| Token     | Logit |
| --------- | ----: |
| Paris     |  18.2 |
| Lyon      |  14.1 |
| Marseille |  13.0 |

Para convertirlos en probabilidades se utiliza **Softmax**.

```python
probs = torch.softmax(next_token_logits, dim=-1)
```

Ahora sí obtenemos una distribución de probabilidad.

---

**Obtener los mejores candidatos**

Podemos inspeccionar qué está pensando el modelo antes de generar un token.

```python
next_token_logits = outputs.logits[:, -1, :]

probs = torch.softmax(next_token_logits, dim=-1)

top_probs, top_ids = torch.topk(probs, k=10)

for prob, token_id in zip(top_probs[0], top_ids[0]):
    print(tokenizer.decode(token_id), prob.item())
```

Esto nos mostrará algo similar a:

```text
 Paris       0.923
 Lyon        0.021
 Marseille   0.008
```

**Tipos de LLMs**

No todos los modelos de lenguaje funcionan de la misma manera.

Actualmente podemos dividirlos en **tres grandes familias**, dependiendo de cómo fueron entrenados y de la arquitectura Transformer que utilizan.

| Tipo                          | Arquitectura      | ¿Genera texto? | Ejemplos                         |
| ----------------------------- | ----------------- | -------------- | -------------------------------- |
| **Encoder (Autoencoder)**     | Solo Encoder      | ❌              | BERT, RoBERTa, DeBERTa, ELECTRA  |
| **Decoder (Autoregresivo)**   | Solo Decoder      | ✅              | GPT, Llama, Qwen, Gemma, Mistral |
| **Encoder-Decoder (Seq2Seq)** | Encoder + Decoder | ✅              | T5, BART, mT5                    |

Cada uno está optimizado para resolver un tipo diferente de problema.


# Modelos AutoEncoder

## Generación base

Los modelos encoder están diseñados para **comprender texto**, no para generarlo.

Reciben toda la secuencia de entrada al mismo tiempo.

Por ejemplo:

```text id="7pzzpa"
The movie was fantastic.
```

El modelo procesa toda la oración en una única pasada y genera una representación contextual para cada token.

```text id="bqqfep"
The         → vector
movie       → vector
was         → vector
fantastic   → vector
.           → vector
```

Una vez obtenidos esos vectores, una pequeña red neuronal (task head) produce la salida deseada.


**Características**

* No generan texto.
* No predicen el siguiente token.
* No utilizan autoalimentación (auto-feed).
* Toda la inferencia ocurre en una única pasada.

**Aplicaciones**

* Clasificación de texto
* Análisis de sentimientos
* Detección de spam
* Named Entity Recognition (NER)
* Question Answering extractivo
* Búsqueda semántica
* Generación de embeddings
* Clustering de documentos
* Sistemas de recomendación

**Modelos principales**

* BERT
* RoBERTa
* DeBERTa
* ELECTRA
* ALBERT
* DistilBERT
* ModernBERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# Modelo BERT entrenado para Masked Language Modeling
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

In [ ]:
# Veamos el vocabulario que maneja BERT
vocab = tokenizer.get_vocab()
print("Tamaño del vocabulario:", len(vocab))

vocab_i = 1010
for i in range(vocab_i, vocab_i+10):
    print(i, tokenizer.convert_ids_to_tokens(i))

Tamaño del vocabulario: 30522
1010 ,
1011 -
1012 .
1013 /
1014 0
1015 1
1016 2
1017 3
1018 4
1019 5


Significado general de los tokens especiales:

| Token         | Purpose                                  |
| ------------- | ---------------------------------------- |
| `<BOS>`       | Beginning Of Sequence                    |
| `<EOS>`       | End Of Sequence                          |
| `<PAD>`       | Padding                                  |
| `<UNK>`       | Unknown token                            |
| `<MASK>`      | Masked token (encoder models)            |
| `<CLS>`       | Classification token                     |
| `<SEP>`       | Separator token                          |
| `<SOS>`       | Start Of Sequence (older Seq2Seq models) |
| `<EOT>`       | End Of Turn (chat models)                |
| `<SYSTEM>`    | System prompt marker (chat models)       |
| `<USER>`      | User message marker                      |
| `<ASSISTANT>` | Assistant message marker                 |


In [ ]:
# Veamos el nombre de los tokens especiales
print(tokenizer.special_tokens_map)
print("ID del token de MASK:", tokenizer.mask_token_id)

{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
ID del token de MASK: 103


In [ ]:
# Tokenizamos el input
text = "The capital of France is [MASK]."

inputs = tokenizer(text, return_tensors="pt")
inputs

{'input_ids': tensor([[ 101, 1996, 3007, 1997, 2605, 2003,  103, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
# Pedimos inferencia al modelo
with torch.no_grad():
    outputs = model(**inputs)
outputs

MaskedLMOutput(loss=None, logits=tensor([[[ -6.4346,  -6.4063,  -6.4097,  ...,  -5.7691,  -5.6326,  -3.7883],
         [-14.0119, -14.7240, -14.2120,  ..., -11.6976, -10.7304, -12.7617],
         [ -9.6561, -10.3125,  -9.7459,  ...,  -8.7782,  -6.6036, -12.6596],
         ...,
         [ -3.7861,  -3.8572,  -3.5644,  ...,  -2.5593,  -3.1093,  -4.3820],
         [-11.6598, -11.4274, -11.9266,  ...,  -9.8772, -10.2103,  -4.7594],
         [-11.7267, -11.7509, -11.8040,  ..., -10.5943, -10.9407,  -7.5151]]]), hidden_states=None, attentions=None)

In [ ]:
# Buscar dónde está el token [MASK]
mask_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
mask_index

tensor([6])

In [ ]:
# Obtener los logits
logits = outputs.logits
print("Shape of logit matrix:", logits.shape)
logits

Shape of logit matrix: torch.Size([1, 9, 30522])


tensor([[[ -6.4346,  -6.4063,  -6.4097,  ...,  -5.7691,  -5.6326,  -3.7883],
         [-14.0119, -14.7240, -14.2120,  ..., -11.6976, -10.7304, -12.7617],
         [ -9.6561, -10.3125,  -9.7459,  ...,  -8.7782,  -6.6036, -12.6596],
         ...,
         [ -3.7861,  -3.8572,  -3.5644,  ...,  -2.5593,  -3.1093,  -4.3820],
         [-11.6598, -11.4274, -11.9266,  ...,  -9.8772, -10.2103,  -4.7594],
         [-11.7267, -11.7509, -11.8040,  ..., -10.5943, -10.9407,  -7.5151]]])

In [ ]:
mask_logits = outputs.logits[0, mask_index, :]
probs = torch.softmax(mask_logits, dim=-1)
probs

tensor([[4.1095e-08, 3.8276e-08, 5.1294e-08,  ..., 1.4016e-07, 8.0858e-08,
         2.2647e-08]])

In [ ]:
# Top 10 candidatos
top_probs, top_ids = torch.topk(probs, k=10)

print("=== Top 10 predicciones ===\n")

for rank, (token_id, prob) in enumerate(zip(top_ids[0], top_probs[0]), start=1):
    token = tokenizer.decode([token_id])

    print(
        f"{rank:2d}. "
        f"{token:<15}"
        f"Probabilidad = {prob.item():.4f}"
    )

# Predicción final
best_token = tokenizer.decode([top_ids[0][0]])

print("\nRespuesta final:")
print(best_token)

# Reconstruir la oración
completed = text.replace(tokenizer.mask_token, best_token)

print("\nOración completa:", completed)

=== Top 10 predicciones ===

 1. paris          Probabilidad = 0.4168
 2. lille          Probabilidad = 0.0714
 3. lyon           Probabilidad = 0.0634
 4. marseille      Probabilidad = 0.0444
 5. tours          Probabilidad = 0.0303
 6. toulouse       Probabilidad = 0.0288
 7. orleans        Probabilidad = 0.0254
 8. nantes         Probabilidad = 0.0228
 9. brest          Probabilidad = 0.0226
10. bordeaux       Probabilidad = 0.0212

Respuesta final:
paris

Oración completa: The capital of France is paris.


## Seguimiento de instrucciones

Para darle intención al modelo y pasar de simple generación a toma de instrucciones se puede reentrenar, afinar o usar prompt engineering para guiar la respuesta.

En este caso usaremos prompt engineering.

In [ ]:
def fill_mask(text, full_text=True):
  inputs = tokenizer(text, return_tensors="pt")

  with torch.no_grad():
    outputs = model(**inputs)

  mask_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
  mask_logits = outputs.logits[0, mask_index, :]
  probs = torch.softmax(mask_logits, dim=-1)

  top_probs, top_ids = torch.topk(probs, k=1)
  best_token = tokenizer.decode([top_ids[0][0]])

  completed = text.replace(tokenizer.mask_token, best_token)

  if full_text:
    return completed
  else:
    return best_token

In [ ]:
print(fill_mask("The review of the movie was [MASK] because the person said: I liked the movie"))
print(fill_mask("The review of the movie was [MASK] because the person said: I hated the movie"))

The review of the movie was positive because the person said: I liked the movie
The review of the movie was negative because the person said: I hated the movie


In [ ]:
# Creamos un wrapper clasificador
def clasify_movie_review_sentiment(raw_review):
  prompt_template = f"The review of the movie was [MASK] because the person said: {raw_review}"
  prediction = fill_mask(prompt_template, full_text=False)
  return prediction

In [ ]:
print(clasify_movie_review_sentiment("I liked the movie"))
print(clasify_movie_review_sentiment("I didn't like the movie"))
print(clasify_movie_review_sentiment("I hated the movie. What were they thinking with that horrible acting?"))

positive
negative
negative


# Modelos Autorregresivos

## Generación base

Son los modelos utilizados actualmente en la mayoría de asistentes conversacionales.

Su objetivo es extremadamente simple:

> **Predecir el siguiente token dado todo el contexto anterior.**

Por ejemplo:

```text id="w1a4tv"
The cat
      │
      ▼
predice " sat"
      │
      ▼
The cat sat
           │
           ▼
predice " on"
           │
           ▼
The cat sat on
```

Este proceso recibe el nombre de **generación autoregresiva**.

**Características**

* Generan texto.
* Predicen un token cada vez.
* Utilizan auto-feed (cada token generado pasa a formar parte del contexto).
* La inferencia consiste en múltiples pasos consecutivos.

**Aplicaciones**

* Chatbots
* Asistentes virtuales
* Traducción mediante prompting
* Programación
* Resúmenes
* Escritura creativa
* Preguntas y respuestas
* Agentes de IA

**Modelos principales**

* GPT
* Llama
* Qwen
* Gemma
* Mistral
* Phi


**¿Cómo genera texto un modelo?**

Los modelos autoregresivos generan **un token a la vez**, entonces para generar una frase completa realiza inferencia sobre su mismo output a modo de auto-feed.

Este proceso recibe el nombre de **generación autoregresiva**.


**Buenas prácticas**

Para la mayoría de aplicaciones es recomendable especificar explícitamente:

```python
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
)
```

Si el objetivo es estudiar el comportamiento interno del modelo, es preferible comenzar con:

```python
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
)
```

De esta forma el modelo siempre seleccionará el token más probable, facilitando el análisis paso a paso de cómo realiza la generación.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
prompt = "Today I'm gonna eat some"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Today I'm gonna eat some fruit and make a little snack. What's that?
A. Fruit cake
B. Fruit juice
C. Fruit salad
Answer:
C

Given that a is a positive number, then 'a < 4' is a ______ condition for


In [ ]:
print("Input shape:", inputs.input_ids.shape)
print("Output shape:", output.shape)
print(output)

Input shape: torch.Size([1, 6])
Output shape: torch.Size([1, 56])
tensor([[15364,   358,  2776, 16519,  8180,  1045, 13779,   323,  1281,   264,
          2632, 39359,    13,  3555,   594,   429,  5267,    32,    13, 43087,
         19145,   198,    33,    13, 43087, 22815,   198,    34,    13, 43087,
         32466,   198, 16141,   510,    34,   271, 22043,   429,   264,   374,
           264,  6785,  1372,    11,  1221,   364,    64,   366,   220,    19,
             6,   374,   264, 32671,  2971,   369]])



**¿Cuándo deja de generar?**

La generación termina cuando ocurre alguna de estas condiciones:

1. El modelo genera el token **EOS** (End Of Sequence).
2. Se alcanza `max_new_tokens`.
3. Se activa algún criterio de parada personalizado.

---

**¿Qué es el token EOS?**

Durante el entrenamiento los documentos terminan con un token especial.

Ejemplo:

```text
The cat sat on the mat.<eos>
```

El modelo aprende que, después del final natural de un texto, debe predecir `<eos>`.

Cuando `generate()` detecta ese token, la generación termina automáticamente.

In [ ]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<|endoftext|>
151643


In [ ]:
# Veamos un caso en donde termine con EOS
prompt = "Finally, please click here for"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Finally, please click here for the final versions of the four presentations.


In [ ]:
print("Input shape:", inputs.input_ids.shape)
print("Output shape:", output.shape)
print(output)

Input shape: torch.Size([1, 6])
Output shape: torch.Size([1, 15])
tensor([[ 23949,     11,   4486,   4205,   1588,    369,    279,   1590,  10795,
            315,    279,   3040,  37380,     13, 151643]])


**¿Cómo decide cuál token generar?**

Existen dos estrategias principales.

**Greedy Decoding**

```python
do_sample=False
```

Siempre selecciona el token más probable.

Si las probabilidades son:

```text
the     42%
a       30%
my      15%
```

siempre elegirá:

```text
the
```

La salida será determinista.

In [ ]:
prompt = "My favorite fruit is the"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=1,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


My favorite fruit is the apple


In [ ]:
# Veamos en base a qué la decisión es greedy
with torch.no_grad():
    outputs = model(**inputs)

print("", outputs.logits.shape)
print("Shape is (batch_size, sequence_length, vocab_size)")
logits = outputs.logits
logits

 torch.Size([1, 5, 151936])
Shape is (batch_size, sequence_length, vocab_size)


tensor([[[13.0625, 11.4375,  0.8125,  ..., -2.5781, -2.5781, -2.5781],
         [13.2500, 11.3125,  9.3750,  ...,  2.0000,  2.0000,  2.0000],
         [16.7500, 12.4375,  9.8125,  ..., -0.5703, -0.5703, -0.5703],
         [ 9.8125, 11.0625,  8.3125,  ..., -1.6016, -1.6016, -1.6016],
         [10.3125, 12.3125,  8.1875,  ..., -0.3496, -0.3496, -0.3496]]],
       dtype=torch.bfloat16)

In [ ]:
# El vocabulario del modelo es
vocab = tokenizer.get_vocab()
vocab_i = 1000
for i in range(vocab_i, vocab_i+10):
    print(i, tokenizer.convert_ids_to_tokens(i))

1000 atus
1001 Th
1002 itle
1003 rit
1004 void
1005 ().
1006 (Ċ
1007 Ġoff
1008 Ġother
1009 Ġ&&


In [ ]:
next_token_logits = logits[:, -1, :]
print("Next token logits:", next_token_logits)
probs = torch.softmax(next_token_logits, dim=-1)
print()
print("Probs:", probs)
print()
print("Top tokens to generate:")
top_probs, top_ids = torch.topk(probs, k=10)
for prob, token_id in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode(token_id)
    print(f"{token!r:15} {prob.item():.4f}")

Next token logits: tensor([[10.3125, 12.3125,  8.1875,  ..., -0.3496, -0.3496, -0.3496]],
       dtype=torch.bfloat16)

Probs: tensor([[1.3530e-05, 1.0014e-04, 1.6168e-06,  ..., 3.1650e-10, 3.1650e-10,
         3.1650e-10]], dtype=torch.bfloat16)

Top tokens to generate:
' apple'        0.1318
' banana'       0.0625
' one'          0.0486
' grape'        0.0430
' pineapple'    0.0430
' strawberry'   0.0430
' orange'       0.0295
' mango'        0.0260
' cherry'       0.0260
' water'        0.0229


In [ ]:
# Check prediction rank of a token
token_id = tokenizer.encode(" bone", add_special_tokens=False)[0]
ranking = torch.argsort(next_token_logits, descending=True)
rank = (ranking == token_id).nonzero(as_tuple=True)[1].item() + 1
print(rank)

2387


**Sampling**

```python
do_sample=True
```

Ahora el modelo muestrea según las probabilidades.

A veces elegirá:

```text
the
```

otras veces:

```text
a
```

Esto hace que la generación sea más variada.

---

**Temperatura**

La temperatura modifica la distribución de probabilidades antes del muestreo.

```python
temperature=0.2
```

Hace al modelo muy conservador.

```python
temperature=1.0
```

No modifica la distribución.

```python
temperature=2.0
```

La hace mucho más uniforme, aumentando la creatividad.

Valores habituales:

```text
0.2   Muy determinista
0.5   Conservador
0.7   Recomendado
1.0   Normal
1.2+  Creativo
```

---

**Parámetros más comunes de `model.generate()`**

| Parámetro | Descripción | Ejemplo |
|-----------|-------------|---------|
| `max_new_tokens` | Número máximo de tokens que el modelo puede generar. No incluye los tokens del prompt. | `max_new_tokens=100` |
| `do_sample` | Si es `True`, el siguiente token se selecciona mediante muestreo según las probabilidades. Si es `False`, siempre elige el token más probable (Greedy Decoding). | `do_sample=True` |
| `temperature` | Controla la aleatoriedad de la generación. Valores bajos producen respuestas más deterministas; valores altos, más creativas. | `temperature=0.7` |
| `top_k` | Limita la selección a los **K** tokens más probables antes de realizar el muestreo. | `top_k=50` |
| `top_p` | Mantiene únicamente los tokens cuya probabilidad acumulada alcanza un porcentaje **P** (Nucleus Sampling). | `top_p=0.9` |
| `repetition_penalty` | Penaliza tokens que ya han aparecido para reducir repeticiones. Valores mayores a `1.0` reducen la repetición. | `repetition_penalty=1.1` |
| `no_repeat_ngram_size` | Impide que el modelo repita secuencias de **N** tokens consecutivos. | `no_repeat_ngram_size=3` |
| `num_return_sequences` | Genera varias continuaciones diferentes para el mismo prompt. Requiere `do_sample=True` para obtener variedad. | `num_return_sequences=5` |
| `use_cache` | Activa el **KV Cache** para acelerar la generación reutilizando cálculos anteriores. Normalmente está activado por defecto. | `use_cache=True` |
| `pad_token_id` | Token utilizado para realizar padding durante la generación. En muchos LLMs se establece igual al `eos_token_id`. | `pad_token_id=tokenizer.eos_token_id` |
| `eos_token_id` | Token que indica el final de la secuencia. Si el modelo lo genera, `generate()` detiene la generación automáticamente. | `eos_token_id=tokenizer.eos_token_id` |

In [ ]:
# Ahora probemos con otra temperatura
def generate_text(text, max_tokens=1, temp=0.7, full_text=True, do_sample=True):
  inputs = tokenizer(text, return_tensors="pt").to(model.device)

  output = model.generate(
      **inputs,
      max_new_tokens=max_tokens,
      do_sample=do_sample,
      temperature=temp,
      repetition_penalty=1.2
  )

  if full_text:
    return tokenizer.decode(output[0], skip_special_tokens=True)
  else:
    # Number of tokens in the prompt
    prompt_length = inputs.input_ids.shape[1]

    # Keep only the generated tokens
    generated_ids = output[0][prompt_length:]

    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
generate_text("My favorite fruit is the", max_tokens=1, temp=0.8)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'My favorite fruit is the apple'

In [ ]:
test_temps = [0.2, 0.8, 1.2, 2.0]
for temp in test_temps:
  print(f"Temperature: {temp}")
  print(generate_text("My favorite fruit is the", max_tokens=1, temp=temp))
  print()

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Temperature: 0.2


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


My favorite fruit is the apple

Temperature: 0.8


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


My favorite fruit is the banana

Temperature: 1.2


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


My favorite fruit is the pineapple

Temperature: 2.0
My favorite fruit is the peach



In [ ]:
test_temps = [0.2, 0.8, 1.2, 2.0]
for temp in test_temps:
  print(f"Temperature: {temp}")
  print(generate_text("Tomorrow is raining. I think I'm going to", max_tokens=20, temp=temp))
  print()

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Temperature: 0.2


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Tomorrow is raining. I think I'm going to have a bad day.
I've been thinking about this for quite some time now, and it's

Temperature: 0.8


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Tomorrow is raining. I think I'm going to miss the movie.
I can't find my umbrella at home, and it's windy outside.

Does

Temperature: 1.2


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Tomorrow is raining. I think I'm going to buy lunch and sit outside at some poolside location on a rainy afternoon with my friends, then watch

Temperature: 2.0
Tomorrow is raining. I think I'm going to cry tears, but my child thinks so as he plays the sky outside their heads! Both ways that



Ahora un ejemplo explícito de una inferencia autorregresiva.

In [ ]:
# Inferencia interna simulada
output = "Yesterday was a good day because"
for i in range(10):
  output = generate_text(output, max_tokens=1, do_sample=False)
  print(output)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Yesterday was a good day because I


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Yesterday was a good day because I had


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

Yesterday was a good day because I had the chance to go on


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Yesterday was a good day because I had the chance to go on an


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Yesterday was a good day because I had the chance to go on an adventure
Yesterday was a good day because I had the chance to go on an adventure with


In [ ]:
# Vista desde el wrapper
generate_text("Yesterday was a good day because", max_tokens=10, do_sample=False)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'Yesterday was a good day because I had the chance to go on an adventure with'

**¿El modelo recuerda conversaciones anteriores?**

No.

Los modelos por default son **completamente stateless**.

Cada llamada a:

```python
model.generate(...)
```

empieza desde cero.

Si primero hacemos:

```text
The sky is
```

y luego:

```text
Cats are
```

la segunda generación **no sabe absolutamente nada** de la primera.

La única forma de mantener memoria es que **nosotros** le volvamos a enviar el historial o reutilicemos el **KV Cache**.


## Seguimiento de instrucciones

Para darle intención al modelo y pasar de simple generación a toma de instrucciones se puede reentrenar, afinar o usar prompt engineering para guiar la respuesta.

En este caso usaremos prompt engineering. Veamos varios ejemplos

**Ejemplo con detección de idioma**

In [ ]:
prompt = "Detect the language of the following phrase: 'Hola, me llamo Carlos'. The language is"
generate_text(prompt, max_tokens=1, temp=0.2)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


"Detect the language of the following phrase: 'Hola, me llamo Carlos'. The language is Spanish"

In [ ]:
def detect_language(phrase):
  prompt = f"Detect the language of the following phrase: '{phrase}'. The language is"
  return generate_text(prompt, max_tokens=1, temp=0.2, full_text=False)

print(detect_language("Je suis bien"))
print(detect_language("Tengo mucha hambre"))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 French
 Spanish


**Ejemplo con traducción automática**

In [ ]:
prompt = "Translate the following phrase to english: 'Hola, me llamo Carlos'. The translation is"
generate_text(prompt, max_tokens=20, temp=0.2)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'Translate the following phrase to english: \'Hola, me llamo Carlos\'. The translation is "Hello, my name is Carlos".'

In [ ]:
def translate_language(phrase):
  prompt = f"Translate the following phrase to english: '{phrase}'. The translation is"
  return generate_text(prompt, max_tokens=20, temp=0.2, full_text=False)

print(translate_language("Hola, me llamo Carlos"))
print(translate_language("Quelle bonne idée!"))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 "Hello, my name is Carlos."
 "What a good idea!"


**Ejemplo como todólogo (QnA)**

In [ ]:
# Sin instrucción el modelo simplemente genera
prompt = "Does hemoglobin carry oxygen or mercury?"
generate_text(prompt, max_tokens=100, temp=0.1)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'Does hemoglobin carry oxygen or mercury? - Quora\nDo you know if there is a way to make the sun shine on your face?\n3 Answers\nBest\nQuora User\n, former Retired Engineer (1954-2007)\nAnswered 6 months ago · Author has 8.9K answers and 7M answer views\nThe Sun shines directly at Earth’s surface.\nIt does not “shine” in any other direction except for its own light source which it emits from within itself as'

In [ ]:
# Con instrucción lo guiamos a que sepa exactamente qué hace más sentido generar y cuándo parar
prompt = """
Question: Does hemoglobin carry oxygen or mercury?.
Answer:
"""
generate_text(prompt, max_tokens=100, temp=0.1)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'\nQuestion: Does hemoglobin carry oxygen or mercury?.\nAnswer:\nHemoglobin carries oxygen.'

In [ ]:
def answer_question(question):
  prompt = f"""
  Question: {question}
  Answer:
  """
  return generate_text(prompt, max_tokens=100, temp=0.2, full_text=False)

print(answer_question("What is the square root of sixty four?"))
print(answer_question("How many electrons does an oxygen atom have?"))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 The question "What is simply the square root of sixty-four?" can be answered by calculating the principal (non-negative) square root of 64. 

The steps are as follows:

1. Identify that we need to find a number \( x \) such that when it is squared, it equals 64.
2. Calculate \( \sqrt{64} = 8 \).

Therefore, the answer is \(\boxed{8}\).
 - Oxygen has six protons and eight neutrons.
   - The number of electrons in a neutral atom is equal to the atomic number, which for oxygen (O) is 8.

Answer:

Assistant: An oxygen atom contains **eight** electrons.


**Ejemplo como programador**

In [ ]:
prompt = """
Build a python script based on this description:
A function that takes a list and returns inly positive numbers.
The answer code with full implementation is:
"""
print(generate_text(prompt, max_tokens=100, temp=0.1))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Build a python script based on this description:
A function that takes a list and returns inly positive numbers.
The answer code with full implementation is:
def only_positive_numbers(numbers):
    return [num for num in numbers if num > 0]
# Example usage: 
numbers = [-1, -2, -3, 4, 5]  
result = only_positive_numbers(numbers)  

print(result)
Output:

[4, 5]

In the provided Python solution, there are several important points to consider when implementing functions. Here's an explanation of each point along with some additional tips.

### Understanding Functions

Functions allow you to


**Ejemplo como chatbot**

In [ ]:
# Generación base de ejemplo
chat = """
PERSONA 1: Hola
PERSONA 2: Hola, como estas?
PERSONA 1: Muy bien, un poco cansado. Tu como estas?
"""
print(generate_text(chat, max_tokens=50, temp=0.1))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



PERSONA 1: Hola
PERSONA 2: Hola, como estas?
PERSONA 1: Muy bien, un poco cansado. Tu como estas?
PERSONA 2: Bien, gracias.
PERSONA 3: ¿Qué te gusta hacer en la playa? 
Persona 4: Me encanta nadar y bailar con los amigos.

Answer the following question:
What is Persona 5


In [ ]:
# Agregamos contexto
chat = """
Eres un CHATBOT de ayuda y estás conversando con PERSONA_1
PERSONA_1: Hola
CHATBOT: Hola, ¿como te puedo ayudar? ¿cómo estás?
PERSONA_1: Muy bien, un poco cansado. Tu como estas?
"""
print(generate_text(chat, max_tokens=50, temp=0.1))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Eres un CHATBOT de ayuda y estás conversando con PERSONA_1
PERSONA_1: Hola
CHATBOT: Hola, ¿como te puedo ayudar? ¿cómo estás?
PERSONA_1: Muy bien, un poco cansado. Tu como estas?
CHATBOT: Bien, gracias por la atención.
PERSONA_1: Pues no me gusta mucho el trabajo que hago aquí en este lugar. Me siento muy solo y sin compañía.
CHATBOT: Sí, es cierto.


In [ ]:
# Creamos un bucle para hacer una simulacion
chat_history = """
Eres un CHATBOT de ayuda y estás conversando con PERSONA.
Este es el historial del chat:
CHATBOT: Hola
"""
while True:
  user_input = input("PERSONA: ")
  if user_input == "exit":
    break
  chat_history += f"PERSONA: {user_input}\n"
  prompt = chat_history + "Tu respuesta es:"
  response = generate_text(prompt, max_tokens=50, temp=0.1, full_text=False)
  chat_history += f"CHATBOT: {response}\n"
  print("CHATBOT:", response)

PERSONA: Hola


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


CHATBOT:  ¡Hola!
El usuario no ha dicho nada, pero te pides una información. Puedes preguntar algo o simplemente decir "¿Qué quieres saber?" para iniciar la conversación.

Si necesitas más detalles sobre algún tema específico,
PERSONA: Me puedes ayudar con algo?


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


CHATBOT:  Sí, ¿qué tipo de asistencia me podrías dar?

Puede que sea útil hacer algunas preguntas a continuación:

1. **Información general**: ¿Cómo puedo ser tu mejor amigo? (Respuesta: Es importante tener en
PERSONA: Me gustaría saber en donde queda la Republica Dominicana


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


CHATBOT:  La República Dominicana se encuentra ubicada al sur de América Latina, cerca de los Estados Unidos. Tiene ciudades como Santo Domingo, San Juan, Nueva York, Miami, etc., dependiendo de su localización exacta dentro de
PERSONA: exit


In [ ]:
chat_history = """Eres un asistente virtual amigable y conversacional.

Reglas:
- Responde como Chatbot.
- Responde únicamente como Chatbot.
- No escribas mensajes de Persona.
- Mantén las respuestas breves 1 oración.
- Usa muchos emojis 😊.
- Sé comprensivo.

Conversación:

Persona: Hola.
Chatbot: ¡Hola! 😊 ¿Cómo estás?
"""

while True:
    user_input = input("Persona: ")

    if user_input.lower() == "exit":
        break

    chat_history += f"\nPersona: {user_input}\nChatbot:"

    response = generate_text(
        chat_history,
        max_tokens=80,
        temp=0.4,
        full_text=False
    ).strip()

    # Keep only the first line in case the model starts inventing
    # another PERSONA turn.
    response = response.split("Persona:")[0].strip()

    print("Chatbot:", response)

    chat_history += " " + response

Persona: Hoy no me siento muy bien.


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Chatbot: Entiendo. Puedo ayudarte a manejar tu salud o explorar algunas opciones para mejorar tus condiciones físicas? 😊
Persona: Hoy me caí de mi bicicleta bastante duro. Qué me recomiendas hacer?


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Chatbot: Para recuperarse, te recomendamos que descanses lo suficiente durante los primeros días después del accidente. También es importante mantener una buena higiene física y comer alimentos ricos en nutrientios. Si tienes dudas sobre cómo proceder exactamente, puedo proporcionarte más detalles 🤔
Persona: exit


# Modelos Encoder-Decoder

Estos modelos combinan las ventajas de ambas arquitecturas.

Constan de dos redes diferentes:

* Un **Encoder**, que comprende completamente el texto de entrada.
* Un **Decoder**, que genera la salida token por token.

**Paso 1: Comprensión**

Supongamos que queremos traducir:

```text id="tq0w9b"
Translate to Spanish:

Hello world
```

El encoder procesa todo el texto de entrada una sola vez.

```text id="cg9bck"
Texto
   │
   ▼
Encoder
   │
   ▼
Representación interna
```

---

**Paso 2: Generación**

El decoder comienza a generar la respuesta.

Inicialmente recibe un token especial de inicio.

```text id="4v6h0t"
<START> Hola mundo <EOS>
```

**Características**

* Generan texto.
* El decoder funciona de forma autoregresiva.
* El encoder solo se ejecuta una vez.
* El decoder puede consultar continuamente la representación creada por el encoder mediante **Cross-Attention**.

**Aplicaciones**

* Traducción automática
* Resumen abstractivo
* Reescritura de texto
* Corrección gramatical
* Parafraseo
* Transformación de documentos

**Modelos principales**

* T5
* BART
* mT5

# Comparación general

La comparación general sería:

| Característica          |    Encoder   |    Decoder   |     Encoder-Decoder     |
| ----------------------- | :----------: | :----------: | :---------------------: |
| Comprende texto         |       ✅      |       ✅      |            ✅            |
| Genera texto            |       ❌      |       ✅      |            ✅            |
| Predice siguiente token |       ❌      |       ✅      |   ✅ (solo el decoder)   |
| Auto-feed               |       ❌      |       ✅      |   ✅ (solo el decoder)   |
| Una sola inferencia     |       ✅      |       ❌      | Encoder: ✅ / Decoder: ❌ |
| Arquitectura            | Solo Encoder | Solo Decoder |    Encoder + Decoder    |

---

**¿Cuál utilizar?**

La elección depende completamente del problema que queremos resolver.

| Objetivo              | Modelo recomendado |
| --------------------- | ------------------ |
| Clasificar documentos | Encoder            |
| Obtener embeddings    | Encoder            |
| Búsqueda semántica    | Encoder            |
| Detección de spam     | Encoder            |
| NER                   | Encoder            |
| Chatbot               | Decoder            |
| Asistente virtual     | Decoder            |
| Generación de código  | Decoder            |
| Escritura creativa    | Decoder            |
| Traducción automática | Encoder-Decoder    |
| Resumen abstractivo   | Encoder-Decoder    |
| Reescritura de texto  | Encoder-Decoder    |

---

**Resumen**

Podemos resumir las tres familias de la siguiente forma:

* **Encoder:** entiende el texto, pero no escribe.
* **Decoder:** escribe texto prediciendo un token tras otro.
* **Encoder-Decoder:** primero entiende el texto completo y luego escribe una nueva secuencia basada en esa comprensión.

Esta diferencia explica por qué modelos como **RoBERTa** son excelentes clasificadores y generadores de embeddings, mientras que modelos como **Qwen** o **GPT** destacan en tareas conversacionales y de generación de texto, y modelos como **T5** son especialmente eficaces en tareas de transformación de texto como traducción o resumen.